# Capstone — mirrors your deployed research paper

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
# --- setup: clone repo in Colab, put work/src on the path, build (or load cached) cutoff frames ---
import os, sys, json, datetime as dt, importlib
REPO_DIR = "/content/FlyRank-ML-Internship"
if os.path.isdir("/content") and not os.environ.get("CAPSTONE_LOCAL"):
    if not os.path.isdir(REPO_DIR):
        os.system(f"git clone https://github.com/joannadulay/FlyRank-ML-Internship.git {REPO_DIR}")
    else:
        os.system(f"git -C {REPO_DIR} pull -q")            # always use the latest code and JSON files from GitHub
    os.chdir(REPO_DIR)
    os.system("pip install -q datasets duckdb scikit-learn huggingface_hub pyarrow")
sys.path.insert(0, "work/src")
importlib.invalidate_caches()
import numpy as np, pandas as pd
import capstone_lib as L
os.makedirs(L.OUT_DIR, exist_ok=True)
plan, frames, TOKEN, LOCAL = L.start()
train, val = frames["train"], frames["val"]          # the sealed test frame is not touched in this notebook
print("cutoffs  train:", [str(t) for t in plan["train"]], "| val:", plan["val"])
print("rows     train:", len(train), "| val:", len(val))

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B / 72.0MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B / 86.2MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B / 90.3MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

... (13 outputs left out)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

cutoffs  train: ['2026-01-11', '2026-02-08', '2026-03-08', '2026-04-05'] | val: 2026-05-03
rows     train: 237571 | val: 85997


## 1. Question

*The research question and the decision it supports.*

**Question:** Can a page's previous 56 days of search and engagement signals help predict whether its search impressions will grow, recover, hold steady, or decline over the next 28 days?

**Decision supported:** which pages a content team should review first.

In [3]:
print(json.dumps(L.CFG, indent=1))

{
 "window_days": 28,
 "min_avail_days": 14,
 "min_impr_B": 100,
 "change_thr": 0.25,
 "dev_end": "2026-05-31",
 "test_end": "2026-06-30",
 "n_train_cutoffs": 4,
 "seed": 42
}


## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

In [4]:
fbm = L.find_month_files(TOKEN, LOCAL)
T = plan["test"]
months = sorted({d.strftime("%Y-%m") for d in pd.date_range(T - dt.timedelta(days=55), T + dt.timedelta(days=28))})
con = L.open_panel(L.fetch_paths(fbm, months, TOKEN, LOCAL))
funnel = L.funnel(con, T)
print("Release: FlyRank/internship-warehouse | table: fact_content_daily_performance | months in release:", list(fbm)[0], "to", list(fbm)[-1])
print("Cutoffs  train:", [str(t) for t in plan["train"]], "val:", plan["val"], "test (sealed):", plan["test"])
print(pd.Series(funnel).to_string())
# Exclusions: rows with gsc_data_available = False are dropped, never zero-filled; pages below the coverage / volume floors are dropped; GA4 gaps stay missing.

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Release: FlyRank/internship-warehouse | table: fact_content_daily_performance | months in release: 2025-01 to 2026-06
Cutoffs  train: ['2026-01-11', '2026-02-08', '2026-03-08', '2026-04-05'] val: 2026-05-03 test (sealed): 2026-06-02
pages_seen         409326
gsc_ok_in_B        139933
and_in_A           118923
and_enough_impr     92357
and_in_C_final      86910


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

See `w05_model` (features, split, baselines) and `w06_validation_audit` (leakage checks). Label rule (impressions per GSC-available day, window C vs B; Recovery = Growth after a fall from A to B), threshold ±25%.

In [5]:
dev = json.load(open("work/outputs/w05_dev_metrics.json"))
audit = json.load(open("work/outputs/w06_audit.json"))
print("Features:", dev["features"])
print("Leakage:", audit["leakage"])

Features: ['log_rate_B', 'd_impr', 'r7', 'ctr_B', 'd_ctr', 'pos_B', 'd_pos', 'cv_B', 'active_share_B', 'avail_share_B', 'ga4_cov_B', 'eng_per_day_B', 'org_per_day_B']
Leakage: {'max_abs_diff': 1.7763568394002505e-15, 'single_feature_max': 0.39982337981319654, 'single_feature_top': 'r7', 'val_pages_seen_in_train': 0.8004232705792063}


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

This is the **one** evaluation on the sealed test cutoff, with the model and baseline frozen in `model_choice.json`.

In [6]:
choice = json.load(open("work/outputs/model_choice.json"))
test, final = L.get_final(plan, frames, choice)
tab = pd.DataFrame({"model": final["model_scores"], **final["baselines"]}).T[["macro_f1", "accuracy"]].round(3)
print(tab.to_string())
print("\nper-class F1 (model):", {k: round(v, 3) for k, v in final["model_scores"]["per_class_f1"].items()})
print(f"\nmodel minus best baseline ({final['best_baseline']}): {final['diff_vs_best_baseline']:+.3f}, 95% interval {final['diff_ci95'][0]:+.3f} to {final['diff_ci95'][1]:+.3f}")

                macro_f1  accuracy
model           0.382726  0.424278
majority        0.101048  0.253285
persistence     0.267575  0.394638
mean_reversion  0.219761  0.282764
recent_slope     0.30612  0.436118
momentum_tree   0.416944  0.459763

per-class F1 (model): {'Growth': 0.37, 'Recovery': 0.326, 'Stable': 0.287, 'Decline': 0.547}

model minus best baseline (momentum_tree): -0.034, 95% interval -0.068 to -0.012


## 5. Limitations

*What this work cannot claim.*

- Observed, directional associations in one panel; not causal, and nothing about how Google ranks or refreshes pages.
- The label is a change in impressions, which also moves with seasonality, demand and algorithm updates.
- Window B sits in both the features and the label denominator (regression to the mean).
- Uneven coverage: pages need enough GSC-available days; GA4 is missing for many page-days.
- One sealed outcome month.

In [7]:
print("class shares in the sealed test set:", final["class_share_test"]); print("pages kept / pages seen:", funnel["and_in_C_final"], "/", funnel["pages_seen"])

class shares in the sealed test set: {'Decline': 0.6043, 'Stable': 0.2533, 'Growth': 0.0789, 'Recovery': 0.0635}
pages kept / pages seen: 86910 / 409326


## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

In [8]:
queue = json.load(open("work/outputs/queue_evidence.json"))
for r in queue["evidence"]["precision_at_k"]: print(f"{r['name']:<48} declined {r['share_declined']:.1%}  lift {r['lift']:.2f}x")
print("base rate:", round(queue["evidence"]["base_decline"], 3))
pd.DataFrame(queue["preview"]).head(10)

Top 100 by risk                                  declined 76.0%  lift 1.26x
Top 500 by risk                                  declined 80.8%  lift 1.34x
Top 1,000 by risk                                declined 79.1%  lift 1.31x
review_decline_risk (top 5%)                     declined 79.2%  lift 1.31x
Naive: biggest recent fall x volume (top 5%)     declined 75.6%  lift 1.25x
Naive: weakest last-week trend x volume (top 5%) declined 87.7%  lift 1.45x
base rate: 0.604


,rank,action,reasons,p_decline,impressions_28d_rounded
0,1,review_decline_risk,short_term_slide;high_volume,0.38,596400
1,2,review_decline_risk,short_term_slide;ctr_below_peers;high_volume;v...,0.91,198300
2,3,review_decline_risk,short_term_slide;ctr_below_peers;high_volume,0.89,194700
3,4,review_decline_risk,high_volume,0.41,360700
4,5,review_decline_risk,falling_impressions;short_term_slide;ctr_below...,0.84,140400
5,6,review_decline_risk,ctr_below_peers;high_volume;volatile,0.77,145000
6,7,review_decline_risk,high_volume;volatile,0.41,263600
7,8,review_decline_risk,short_term_slide;high_volume,0.79,128000
8,9,review_decline_risk,falling_impressions;short_term_slide;high_volume,0.67,148300
9,10,review_decline_risk,short_term_slide;position_worsening;ctr_below_...,0.73,123400


## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

Charts are drawn here and embedded in the page as images, so the page is a single self-contained file. Deploy: push, then in the repo **Settings → Pages → Source: GitHub Actions** (the workflow in `.github/workflows/pages.yml` publishes `site/`).

In [9]:
import io, base64, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
sys.path.insert(0, "work/src")
import build_paper

def b64(fig):
    buf = io.BytesIO(); fig.savefig(buf, format="png", dpi=140, bbox_inches="tight"); plt.close(fig)
    return base64.b64encode(buf.getvalue()).decode()

figs = {}
names = ["model"] + list(final["baselines"]); vals = [final["model_scores"]["macro_f1"]] + [v["macro_f1"] for v in final["baselines"].values()]
fig, ax = plt.subplots(figsize=(6, 3)); ax.barh([n.replace("_", " ") for n in names][::-1], vals[::-1], color="#1f5f8b")
ax.set_xlabel("macro F1 (sealed test cutoff)"); [ax.text(v, i, f" {v:.3f}", va="center") for i, v in enumerate(vals[::-1])]
figs["compare"] = b64(fig)

C = np.array(final["model_scores"]["confusion"], dtype=float); C = C / C.sum(1, keepdims=True).clip(1)
fig, ax = plt.subplots(figsize=(4.6, 3.8)); ax.imshow(C, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(4), L.CLASSES, rotation=30); ax.set_yticks(range(4), L.CLASSES); ax.set_xlabel("predicted"); ax.set_ylabel("observed")
[ax.text(j, i, f"{C[i, j]:.2f}", ha="center", va="center", color="black" if C[i, j] < .6 else "white") for i in range(4) for j in range(4)]
figs["confusion"] = b64(fig)

imp = pd.Series(dev["importance"]).sort_values().tail(8)
fig, ax = plt.subplots(figsize=(6, 3.2)); ax.barh(imp.index, imp.values, color="#1f5f8b"); ax.set_xlabel("drop in macro F1 when shuffled")
figs["importance"] = b64(fig)

ev = queue["evidence"]; labs = [r["name"] for r in ev["precision_at_k"]]; shares = [r["share_declined"] for r in ev["precision_at_k"]]
fig, ax = plt.subplots(figsize=(7, 3.2)); ax.barh(labs[::-1], shares[::-1], color="#1f5f8b"); ax.axvline(ev["base_decline"], color="#c0392b", ls="--")
ax.set_xlabel("share that actually declined (dashed = all pages)")
figs["lift"] = b64(fig)

REPO_URL = "https://github.com/joannadulay/FlyRank-ML-Internship"
allf = pd.concat([train, val, test[train.columns]], ignore_index=True)
regime = {str(k.date()): g.y.value_counts(normalize=True).round(4).to_dict() for k, g in allf.groupby("cutoff")}
top2 = float(allf.client.value_counts(normalize=True).head(2).sum())
bt_path = "work/outputs/w05_backtest.json"
extra = dict(regime=regime, top2_client_share=top2, n_clients=int(allf.client.nunique()),
             backtest=json.load(open(bt_path)) if os.path.exists(bt_path) else None)
build_paper.build("site/index.html", final, dev, audit, queue, figs, funnel, plan, L.CFG, REPO_URL, extra)
os.makedirs("submission", exist_ok=True)
open("submission/paper_url.txt", "w").write("https://joannadulay.github.io/FlyRank-ML-Internship/\n")
print("wrote site/index.html and submission/paper_url.txt")

wrote site/index.html and submission/paper_url.txt


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.